# GPU Dask Cluster Demo

In this notebook, we show how you can do your work on a temporary cluster of GPUs, just as we did with a cluster of CPUs in the "HelioCloud Storage + Burst + SDO" notebook (SDO_Demo.ipynb). The GPU cluster is created using Dask, a tool that allows us to scale up our compute based on the size of the problem we want to solve. 

In [1]:
import dask
#from dask.distributed import Client
from distributed import Client
from dask_gateway import Gateway, GatewayCluster

import os
import pynvml
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 
import tensorflow as tf
import numpy as np
import time

In the cell below, we set the options for our cluster. We request the workers on the cluster to use HelioCloud's TensorFlow image. As you may recall from the SDO Demo notebook, a CPU cluster can have up to 100 workers on a HelioCloud instance, allowing you to define how many workers you need for your work.

In the case of a GPU cluster, best practice is to tune the cluster options to the type of instance we are using. For the number of workers, we should use the number of cores on the underlying instance. The default HelioCloud instance only has four cores, so we should set `options.worker_cores = 4`. When we set the worker memory, it is recommended to use about 7/8ths of the underlying instance memory. The HelioCloud instance memory is usually 16 GB, so we should set `options.worker_memory = 14`.

Fortunately, the HelioCloud DaskHub server you are working from already has a default profile with these values set, which we have named "gpu-xlarge." Instead of explicitly setting the number of worker cores and memory, we just include the line `options.profile='gpu-xlarge'`.

In [2]:
gateway = Gateway()
options = gateway.cluster_options()

options.image = 'public.ecr.aws/q3h7b4o8/heliocloud/helio-daskhub-mltf:2025.01.29'
options.profile='gpu-xlarge'

Now that we have the cluster options set, we initialize the cluster itself. In the next cell, several things are happening. 

First, we create the cluster and the client that allows us to interface with the cluster. Second, we manually scale the cluster to use a specific number of workers. Since we set `options.worker_cores` to 4, we can scale it to a maximum of 4 workers.

As you might have seen in previous notebooks, we can also automatically scale the cluster, by replacing the line `cluster.scale(n_workers)` with the line `cluster.adapt(minimum=1, maximum=n_workers)`. This option will allow the cluster to scale up and down automatically depending on the compute power required by the tasks you send it. For the sake of the demonstration though, we use the `cluster.scale` option with specifically 3 workers to easily show that the cluster is doing exactly what we expect it to do.

In [3]:
now = time.time()
cluster = gateway.new_cluster(options)
client = cluster.get_client()

n_workers = 3
cluster.scale(n_workers)

print("Cluster initialized in ", (time.time()-now)/60.0, " minutes. Now waiting for GPU workers to spin up.")

def get_nvidia_driver_version():
    return pynvml.nvmlSystemGetDriverVersion(), len(tf.config.list_physical_devices('GPU'))

time_elapsed = 0

while True:
    gpus_dict = client.run(get_nvidia_driver_version) 
 
    # If no GPUs have been spun up
    if gpus_dict:
        total_min = time_elapsed // 60
        total_sec = time_elapsed % 60
        print(f"At least one GPU has been started. Displaying cluster UI. Approx time elapsed: {total_min} minutes and {total_sec} seconds.")
        break  
    else:
        time_elapsed += 15
        time.sleep(15)

    if time_elapsed % 60 == 0:
        min_elapsed = time_elapsed / 60
        print(f"{min_elapsed} minutes elapsed")

cluster

Cluster initialized in  3.679679298400879  minutes. Now waiting for GPU workers to spin up.
1.0 minutes elapsed
2.0 minutes elapsed
3.0 minutes elapsed
4.0 minutes elapsed
At least one GPU has been started. Displaying cluster UI. Approx time elapsed: 4 minutes and 0 seconds.


Now we will give the cluster a computationally heavy mathematical task. The below function `do_tensor_math` will serve this purpose. It multiplies two user-defined matricies, and does so as many times as the user requests.

In [4]:
def do_tensor_math(array_a, array_b, num_loops):
    import os
    # Set log level to 3 to supress INFO and WARNING messages
    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 
    import tensorflow as tf
    
    num_gpus = len(tf.config.list_physical_devices('GPU'))

    tf.debugging.set_log_device_placement(True)
    
    a = tf.constant(array_a)
    b = tf.constant(array_b)
    
    for i in range(num_loops):
        c = tf.matmul(a, b)
        
    return c

For our purposes, we will give the function two very large matricies, and have it multiply them together many hundreds of times. Below, we define the matricies and the number of times to perform the multiplication (for now, we set this to 500). Then we can run the next cell, which submits the `do_tensor_math` job to the client to run on the cluster.

In [5]:
array_a = np.random.rand(4000,6000).astype(np.float32)
array_b = np.random.rand(6000,4000).astype(np.float32)

num_loops = 500

In [6]:
%%time
a_scatter = client.scatter(array_a)
b_scatter = client.scatter(array_b)
c = client.submit(do_tensor_math, a_scatter, b_scatter, num_loops)

CPU times: user 328 ms, sys: 44.7 ms, total: 372 ms
Wall time: 990 ms


This should take around a second. How long would it take to run the same computation on a single GPU, like we do in CPU_vs_GPU_Speed_Test.ipynb? Let's try it.

In [7]:
%%time
with tf.device('/GPU:0'):
    c = do_tensor_math(array_a, array_b, num_loops)

CPU times: user 283 ms, sys: 403 ms, total: 686 ms
Wall time: 709 ms


Obviously, this is much faster with the single GPU than the GPU cluster. Now lets increase the number of multiplications we perform to 1,000.

In [8]:
num_loops = 1000

In [9]:
%%time
a_scatter = client.scatter(array_a)
b_scatter = client.scatter(array_b)
c = client.submit(do_tensor_math, a_scatter, b_scatter, num_loops)

CPU times: user 348 ms, sys: 25.3 ms, total: 374 ms
Wall time: 866 ms


In [10]:
%%time
with tf.device('/GPU:0'):
    c = do_tensor_math(array_a, array_b, num_loops)

CPU times: user 13.2 s, sys: 112 ms, total: 13.3 s
Wall time: 13.3 s


After running the above three cells, you will hopefully find that the single GPU still takes less time to run than the GPU cluster. Let's increase the number of multiplications again to 1,200. The cell that runs our function on the single GPU will take a little longer this time (though still less than a minute).

In [19]:
num_loops = 100

In [20]:
%%time
a_scatter = client.scatter(array_a)
b_scatter = client.scatter(array_b)
c = client.submit(do_tensor_math, a_scatter, b_scatter, num_loops)

CPU times: user 339 ms, sys: 52.7 ms, total: 392 ms
Wall time: 818 ms


In [21]:
%%time
with tf.device('/GPU:0'):
    c = do_tensor_math(array_a, array_b, num_loops)

CPU times: user 34.6 s, sys: 113 ms, total: 34.7 s
Wall time: 34.7 s


You can see that, when the computation demand is not so extreme, the single GPU is as fast or faster than the GPU cluster. As we increase the number of times to perform the matrix multiplication, the GPU cluster far outperforms the single GPU in speed. 

Feel free to increase the `num_loops` value on your own, and test the single GPU vs. the GPU cluster. You will find that the GPU cluster can easily handle the increased number of matrix multiplications, while the single GPU only slows down further and further.

It is worth considering how computationally intensive your task might be. It may actually take less time to simply use the single GPU core made available by HelioCloud's TensorFlow server image. While the time to execute a task on the GPU cluster may be extremely fast, you should also take into account the amount of time required for the cluster to initate, and for the GPU workers to spin up. 

When you are finished, go ahead and shut the cluster down.

In [ ]:
cluster.shutdown()